In [1]:
! pip install trl 

In [2]:
import os 
# Disable progress bars to avoid Jupyter context errors
os.environ['HF_DATASETS_DISABLE_PROGRESS_BAR'] = '1'
import re          
import random      
import logging     
import warnings    
from typing import List, Dict, Optional, Tuple  
from dataclasses import dataclass, field  

import numpy as np  
import torch 
from trl import (
    GRPOConfig,   
    GRPOTrainer 
)
from transformers import (
    AutoTokenizer,  
    AutoModelForCausalLM,    
)

from datasets import load_dataset, load_from_disk

warnings.filterwarnings('ignore')
random.seed(42)    
np.random.seed(42)
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ libraries imported")

✅ libraries imported


### Creating and TrainerConfig Class 

In [3]:
 #============================================
# Define Training Configuration Class
# ============================================

@dataclass
class TrainingConfig:
    """
    Configuration for GRPO training.
    
    This class holds all the settings (hyperparameters) for training.
    Think of it as a control panel with all the knobs and switches.
    
    Each parameter has:
    - A default value (recommended value)
    - A description (what it does)
    - A type (what kind of value it expects)
    """
    
    # ========== MODEL SETTINGS ==========
    # Which model to use and where to save it
    
    model_name: str = field(
        default="Qwen/Qwen3-0.6B",
        metadata={"help": "The pre-trained model to start with."}
    )
    
    output_dir: str = field(
        default="./grpo_finetuned_model",
        metadata={"help": "Where to save the trained model. Like a 'Save As' location."}
    )
    
    # ========== TRAINING DURATION ==========
    # How long to train for
    
    num_train_epochs: int = field(
        default=5,
        metadata={"help": "How many times to go through the training data. More = more learning."}
    )
    
    # ========== BATCH SETTINGS ==========
    # How many examples to process at once
    
    per_device_train_batch_size: int = field(
        default=2,
        metadata={"help": "How many problems to process at once. Limited by GPU memory."}
    )
    
    gradient_accumulation_steps: int = field(
        default=32,
        metadata={"help": "Accumulate gradients over multiple batches. Simulates larger batch size."}
    )
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps = 64
    
    # ========== LEARNING SETTINGS ==========
    # How fast the model learns
    
    learning_rate: float = field(
        default=5e-6,  # 0.000005 in decimal
        metadata={"help": "How big of a step to take when learning. Too high = unstable."}
    )
    
    # ========== GRPO SPECIFIC SETTINGS ==========
    # Settings unique to GRPO algorithm
    
    num_generations: int = field(
        default=12,
        metadata={"help": "How many different answers to generate per question. More = better comparison."}
    )
    
    temperature: float = field(
        default=0.8,
        metadata={"help": "Controls randomness. 0.0 = always same answer, 1.0 = very random."}
    )
    
    max_new_tokens: int = field(
        default=400,
        metadata={"help": "Maximum length of generated answers. Needs to be long enough for full solutions."}
    )
    # ========== DATA SETTINGS ==========
    # How to handle the dataset
    
    max_prompt_length: int = field(
        default=512,
        metadata={"help": "Maximum length of input questions in tokens."}
    )
    
    train_split_ratio: float = field(
        default=0.8,
        metadata={"help": "What fraction of data to use for training (rest for validation)."}
    )
    
    # ========== MONITORING SETTINGS ==========
    # How often to check progress
    
    eval_steps: int = field(
        default=20,
        metadata={"help": "Evaluate model every N steps to check progress."}
    )
    
    save_steps: int = field(
        default=20,
        metadata={"help": "Save a checkpoint every N steps (for recovery if training stops)."}
    )
    
    logging_steps: int = field(
        default=20,
        metadata={"help": "Log training metrics every N steps."}
    )
    
    save_total_limit: int = field(
        default=3,
        metadata={"help": "Keep only the N most recent checkpoints to save disk space."}
    )
    
    # ========== OTHER SETTINGS ==========
    seed: int = field(
        default=42,
        metadata={"help": "Random seed for reproducibility."}
    )
    
    use_8bit: bool = field(
        default=False,
        metadata={"help": "Load model in 8-bit mode to save memory (slightly less accurate)."}
    )

print("✅ Configuration class defined!")

✅ Configuration class defined!


In [4]:
# ============================================
# Create and Display Configuration
# ============================================

# Create an instance of your configuration
# This uses all the default values defined above
config = TrainingConfig()

# Display all configuration values
print("TRAINING CONFIGURATION")
print("="*50)

# Group settings by category for easier reading
print("\nModel Settings:")
print(f"  Model: {config.model_name}")
print(f"  Output directory: {config.output_dir}")

print("\nTraining Duration:")
print(f"  Epochs: {config.num_train_epochs}")
print(f"  Batch size per device: {config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {config.gradient_accumulation_steps}")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")

print("\nGRPO Settings:")
print(f"  Generations per prompt: {config.num_generations}")
print(f"  Temperature: {config.temperature}")
print(f"  Max new tokens: {config.max_new_tokens}")
print(f"  Learning rate: {config.learning_rate}")

print("\nMonitoring:")
print(f"  Evaluate every: {config.eval_steps} steps")
print(f"  Save every: {config.save_steps} steps")
print(f"  Log every: {config.logging_steps} steps")

# Calculate approximate training time
print("\nEstimated Training Info:")
print(f"  This configuration will generate {config.num_generations} answers per question")
print(f"  The model will learn by comparing these answers")

TRAINING CONFIGURATION

Model Settings:
  Model: Qwen/Qwen3-0.6B
  Output directory: ./grpo_finetuned_model

Training Duration:
  Epochs: 5
  Batch size per device: 2
  Gradient accumulation: 32
  Effective batch size: 64

GRPO Settings:
  Generations per prompt: 12
  Temperature: 0.8
  Max new tokens: 400
  Learning rate: 5e-06

Monitoring:
  Evaluate every: 20 steps
  Save every: 20 steps
  Log every: 20 steps

Estimated Training Info:
  This configuration will generate 12 answers per question
  The model will learn by comparing these answers


### Load the GSM8K Dataset

In [24]:
dataset = load_dataset('openai/gsm8k',"main")
print("Dataset Loaded !!")
print(dataset)
print("=="*10)
print("training rows:" ,dataset["train"].num_rows)
print("testing rows:" ,dataset["test"].num_rows)

Dataset Loaded !!
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
training rows: 7473
testing rows: 1319


### Creating reward Function 
#### How It Works

* Extract the numerical answer from the text
* Compare with the correct answer
* Check for partial credit (showing work, being close)
* Assign a reward score

In [5]:
# GSM8KRewardSignal class definition
class GSM8KRewardSignal:
    """
    BASELINE Reward Model for GSM8K (Simplified Version)
    
    This is a basic reward model with only 3 categories:
    - Correct answer: 1.0
    - Wrong answer: 0.0  
    - Unparseable: 0.0
    
    ⚠️ LIMITATIONS: This simple reward signal makes GRPO training difficult!
    The lack of partial credit means the model gets no signal for improvements.
    """
    
    def __init__(self):
        self.logger = logging.getLogger(__name__)

## later it will gets updated

In [6]:
### Extract Numerical Answers from model response 

def extract_numerical_answer(self,text:str) ->Optional[float]:
    """
    Extract the numerical answer from model's response.

    This function looks for numbers in different formats:
    - #### 42 (GSM8K format)
    - "The answer is 42"
    - "= 42"
    - Just finds the last number if nothing else works

    Args:
        text: The model's generated response

    Returns:
        The extracted number, or None if no number found
    """
    ## First check for gsm8k format (#### answers)  
    if "####" in text:
        answer = text.split("####")[-1].strip()
        answer = answer.replace(",","").replace("$","") # to remove the numerical value 
        try:
            return float(answer)
        except:
            pass

    ## trying other regex patterns 
    patterns = [
        r"(?:The answer is|answer:|Answer:)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"(?:equals?|=)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"(?:total|sum|result|Total|Final answer)\s*(?:is|:|=)?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"Therefore,?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern,text,re.IGNORECASE | re.MULTILINE)
        if matches:
            try:
                clean_match = matches[-1].replace("$","").replace(",","")
                return float(clean_match) # the single numerical value 
            except ValueError:
                continue

    ## finding last number in the text 
    numbers = re.findall(r'[+-]?\d+(?:\.\d+)?',text)
    if numbers:
        try:
            return float(numbers[-1])
        except ValueError:
            pass

    return None 

## adding as function in singnal class 
GSM8KRewardSignal.extract_numerical_answer = extract_numerical_answer

In [7]:
# ============================================
# UNIT TEST: - Extract Numerical Answer
# ============================================

def test_extract_numerical_answer():
    """Unit test for the extract_numerical_answer method."""
    print("🧪 Testing Exercise 1: Extract Numerical Answer")
    print("="*50)
    
    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()
    
    # Test cases: (input_text, expected_output, description)
    test_cases = [
        # Standard GSM8K format (already works in base implementation)
        ("The calculation is 5 + 3 = 8 #### 8", 8.0, "GSM8K standard format"),
        
        # Common answer phrases (MUST work after student implementation)
        ("The answer is 42", 42.0, "Common phrase: 'The answer is'"),
        ("Answer: 100", 100.0, "Common phrase: 'Answer:'"),
        ("equals 25", 25.0, "Common phrase: 'equals'"),
        ("total is 15.5", 15.5, "Common phrase: 'total is'"),
        ("Therefore, 7", 7.0, "Common phrase: 'Therefore,'"),
        
        # Numbers with formatting (MUST handle these)
        ("The answer is $1,234.56", 1234.56, "Dollar sign and commas"),
        ("Total: -45", -45.0, "Negative number"),
        ("equals 0.003", 0.003, "Small decimal"),
        
        # Fallback to last number (minimum requirement)
        ("First we have 10, then 20, finally 30", 30.0, "Last number fallback"),
        ("No clear answer but mentions 99", 99.0, "Last number in text"),
    ]
    
    passed = 0
    failed = 0
    
    for text, expected, description in test_cases:
        try:
            result = test_model.extract_numerical_answer(text)
            
            if result is None and expected is not None:
                print(f"❌ FAILED: {description}")
                print(f"   Input: '{text}'")
                print(f"   Expected: {expected}, Got: None")
                failed += 1
            elif result is not None and expected is None:
                print(f"❌ FAILED: {description}")
                print(f"   Input: '{text}'")
                print(f"   Expected: None, Got: {result}")
                failed += 1
            elif result is not None and abs(result - expected) > 0.01:
                print(f"❌ FAILED: {description}")
                print(f"   Input: '{text}'")
                print(f"   Expected: {expected}, Got: {result}")
                failed += 1
            else:
                print(f"✅ PASSED: {description}")
                passed += 1
                
        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1
    
    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")
    
    if failed == 0:
        print("🎉 All tests passed! Exercise 1 is complete.")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements:")
        print("- Extract answers from common phrases (The answer is, equals, total is)")
        print("- Handle numbers with commas and dollar signs")
        print("- Support negative numbers and decimals")
        print("- Fallback to extracting the last number in text")
    
    return passed == len(test_cases)

# Run the test
test_extract_numerical_answer()

🧪 Testing Exercise 1: Extract Numerical Answer
✅ PASSED: GSM8K standard format
✅ PASSED: Common phrase: 'The answer is'
✅ PASSED: Common phrase: 'Answer:'
✅ PASSED: Common phrase: 'equals'
✅ PASSED: Common phrase: 'total is'
✅ PASSED: Common phrase: 'Therefore,'
✅ PASSED: Dollar sign and commas
✅ PASSED: Negative number
✅ PASSED: Small decimal
✅ PASSED: Last number fallback
✅ PASSED: Last number in text

Results: 11/11 passed, 0/11 failed
🎉 All tests passed! Exercise 1 is complete.


True

In [8]:
###  Analyze Response Quality Indicators
def analyze_response_quality(self,response:str) -> Dict[str,any]:
    """
    Analyze the quality indicators of a response.

    This method examines various aspects of the response to determine
    its quality, even if the final answer is wrong or unparseable.

    Args:
        response: The model's generated response

    Returns:
        Dictionary with quality indicators:
        - has_calculation: Boolean, if response contains math operations
        - has_steps: Boolean, if response shows multiple steps
        - has_reasoning: Boolean, if response uses reasoning words
        - has_numbers: Boolean, if response contains numbers
        - response_length: Int, character count
        - sentence_count: Int, number of sentences
    """
    ## check for math operators -> show steps and reasoing
    calculation_indicators = ['=', '+', '-', '*', '/', 'multiply', 'divide', 'add', 'subtract']
    # has_calculations = True , if any indicators/operators in response
    has_calculation = any(op in response for op in calculations_indication) # 1/0

    # check for steps 
    sentences = [s.strip() for s in re.split(r'[.!?]+',response) if s.strip()]
    sentence_count = len(sentences)
    # has_steps = true,if thier are atleast two sentence / length > 200 
    has_steps =  (sentence_count >= 2) or ("step" in response.lower())

    # check for reasoning words 
    reasoning_words = ['therefore', 'because', 'since', 'so', 'thus', 'hence', 
                        'first', 'then', 'next', 'finally', 'must', 'need'] 
    # has_reasoning = true , if any word in response lower  
    has_reasoning = any(op in response.lower() for op in reasoning_words)

    # check for numbers 
    has_numbers = bool(re.search(r'\d',response))
    # response length
    response_length = response.__len__()
    quality = {
        'has_calculation': has_calculation,
        'has_steps': has_steps,
        'has_reasoning': has_reasoning,
        'has_numbers': has_numbers,
        'response_length': response_length,
        'sentence_count': sentence_count
    }

    return quality 


# GRADED CELL: exercise 2

def analyze_response_quality(self, response: str) -> Dict[str, any]:
    """
    Analyze the quality indicators of a response.

    This method examines various aspects of the response to determine
    its quality, even if the final answer is wrong or unparseable.

    Args:
        response: The model's generated response

    Returns:
        Dictionary with quality indicators:
        - has_calculation: Boolean, if response contains math operations
        - has_steps: Boolean, if response shows multiple steps
        - has_reasoning: Boolean, if response uses reasoning words
        - has_numbers: Boolean, if response contains numbers
        - response_length: Int, character count
        - sentence_count: Int, number of sentences
    """

    # Check for calculations (math operators and related words)
    calculation_indicators = ['=', '+', '-', '*', '/', 'multiply', 'divide', 'add', 'subtract'] 
    # set has_calculation to True if any of the indicators are in response.lower()
    has_calculation = any(op in response for op in calculation_indicators)

    # Check for steps (multiple sentences or explicit step markers)
    sentences = [s.strip() for s in re.split(r'[.!?]+', response) if s.strip()] 
    sentence_count = len(sentences)
    # Set has_steps to true if there are at least two sentences or if the word 'step' is in response.lower()
    has_steps = (sentence_count >= 2 ) or ("step" in response.lower())

    # Check for reasoning words
    # These words indicate logical reasoning or step-by-step thinking
    reasoning_words = ['therefore', 'because', 'since', 'so', 'thus', 'hence', 
                        'first', 'then', 'next', 'finally', 'must', 'need'] 
    # Set has_reasoning to True if ANY of these reasoning words appear in the response
    # Use any() with a generator expression to check if any word is in response.lower()
    has_reasoning = any(op in response for op in reasoning_words) 

    # Check for numbers
    has_numbers = bool(re.search(r'\d', response)) 

    # Response length
    response_length = response.__len__()

    quality = {
        'has_calculation': has_calculation,
        'has_steps': has_steps,
        'has_reasoning': has_reasoning,
        'has_numbers': has_numbers,
        'response_length': response_length,
        'sentence_count': sentence_count
    }

    return quality

# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.analyze_response_quality = analyze_response_quality

In [9]:
# ============================================
# UNIT TEST:  - Analyze Response Quality
# ============================================

def test_analyze_response_quality():
    """Unit test for the analyze_response_quality method."""
    print("🧪 Testing Exercise 2: Analyze Response Quality")
    print("="*50)
    
    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()
    
    # Test cases with expected quality indicators
    test_cases = [
        # Test 1: Response with calculations and steps
        {
            "response": "First, let's add 5 + 3 = 8. Then multiply by 2 to get 16.",
            "description": "Response with math operations and reasoning",
            "expected": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": True,
                "has_reasoning": True,  # Changed from has_reasoning_words
                "min_length": 30,
                "min_sentences": 1
            }
        },

        # Test 2: Simple answer without work
        {
            "response": "42",
            "description": "Single number response",
            "expected": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "has_reasoning": False,
                "max_length": 10,
                "max_sentences": 1
            }
        },

        # Test 3: Detailed step-by-step solution
        {
            "response": "Step 1: Calculate 10 * 5 = 50. Step 2: Subtract 15 from 50 to get 35. Finally, divide by 7 for the answer.",
            "description": "Numbered steps with calculations",
            "expected": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": True,
                "has_reasoning": True,
                "min_length": 50,
                "min_sentences": 2
            }
        },

        # Test 4: "I don't know" response
        {
            "response": "I don't know",
            "description": "Give-up response",
            "expected": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": False,
                "has_reasoning": False,
                "max_length": 20,
                "max_sentences": 1
            }
        },

        # Test 5: Response with word-based operations
        {
            "response": "We need to multiply twelve by three and then add seven.",
            "description": "Word-based math operations",
            "expected": {
                "has_calculation": True,
                "has_steps": False,
                "has_numbers": False,  # Changed: No digit characters in response
                "has_reasoning": True,
                "min_length": 20,
                "min_sentences": 1
            }
        }
    ]
    
    passed = 0
    failed = 0
    
    for test_case in test_cases:
        response = test_case["response"]
        description = test_case["description"]
        expected = test_case["expected"]
        
        try:
            quality = test_model.analyze_response_quality(response)
            
            test_passed = True
            failures = []
            
            # Check boolean indicators
            for key in ["has_calculation", "has_steps", "has_numbers", "has_reasoning_words"]:
                if key in expected:
                    if quality.get(key) != expected[key]:
                        failures.append(f"{key}: expected {expected[key]}, got {quality.get(key)}")
                        test_passed = False
            
            # Check response length
            if "min_length" in expected and quality.get("response_length", 0) < expected["min_length"]:
                failures.append(f"response_length: expected >= {expected['min_length']}, got {quality.get('response_length', 0)}")
                test_passed = False
            if "max_length" in expected and quality.get("response_length", 0) > expected["max_length"]:
                failures.append(f"response_length: expected <= {expected['max_length']}, got {quality.get('response_length', 0)}")
                test_passed = False
            
            # Check sentence count
            if "min_sentences" in expected and quality.get("sentence_count", 0) < expected["min_sentences"]:
                failures.append(f"sentence_count: expected >= {expected['min_sentences']}, got {quality.get('sentence_count', 0)}")
                test_passed = False
            if "max_sentences" in expected and quality.get("sentence_count", 0) > expected["max_sentences"]:
                failures.append(f"sentence_count: expected <= {expected['max_sentences']}, got {quality.get('sentence_count', 0)}")
                test_passed = False
            
            if test_passed:
                print(f"✅ PASSED: {description}")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Response: '{response[:50]}...' " if len(response) > 50 else f"   Response: '{response}'")
                for failure in failures:
                    print(f"   - {failure}")
                failed += 1
                
        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1
    
    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")
    
    if failed == 0:
        print("🎉 All tests passed!")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements:")
        print("- Detect math operations (symbols AND words like 'multiply', 'add')")
        print("- Identify reasoning words ('first', 'then', 'next', 'finally')")
        print("- Detect step indicators ('step', numbered lists)")
        print("- Count sentences and measure response length")
        print("- Check for presence of numbers")
    
    return passed == len(test_cases)

# Run the test
test_analyze_response_quality()

🧪 Testing Exercise 2: Analyze Response Quality
✅ PASSED: Response with math operations and reasoning
✅ PASSED: Single number response
✅ PASSED: Numbered steps with calculations
✅ PASSED: Give-up response
✅ PASSED: Word-based math operations

Results: 5/5 passed, 0/5 failed
🎉 All tests passed!


True

In [10]:
### Reward Unparseable Responses
### model may contain numerical answer but model might provide and detailed explanation without final answer 
### eg: "i dont know"

def compute_unparseable_reward(self, response: str, correct_answer: float,
                                quality: Dict[str, any], question: str = None) -> float:
    """
    Compute reward for unparseable responses (no numerical answer found).

    Instead of giving 0.0 to all unparseable responses, give partial
    credit based on the quality of the attempt.

    Args:
        response: The model's generated response
        correct_answer: The correct numerical answer
        quality: Quality indicators from analyze_response_quality

    Returns:
        Reward between 0.0 and 0.3
    """
    response_length = quality.get('response_length', 0)
    has_calculation = quality.get('has_calculation', False)
    has_steps = quality.get('has_steps', False)
    has_numbers = quality.get('has_numbers', False)

    ## very short respose
    if response_length < 20 :
        return 0.0
    ## base reward for any attempt
    reward = 0.05

    # If response has calculations → reward plus 0.05
    if has_calculation:
        reward += 0.05

    # If response has steps → reward plus 0.05
    if has_steps:
        reward += 0.05

    # If response has numbers → reward plus 0.05
    if has_numbers:
        reward += 0.05

    # If response is long (>100 chars) → reward plus 0.05
    if response_length > 100:
        reward += 0.05

    # If response is very long (>200 chars) → reward plus another 0.05
    if response_length > 200:
        reward += 0.05

    self.logger.info(f"⚪ UNPARSEABLE RESPONSE:")
    self.logger.info(f"  Response: {response[:200]}...")
    self.logger.info(f"  Expected: {correct_answer}")
    self.logger.info(f"  Reward: {reward}")

    # Cap reward at 0.3
    return min(0.3, reward) # essential - we dosent want it to go to below 

# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.compute_unparseable_reward = compute_unparseable_reward

In [11]:
# ============================================
# UNIT TEST: Compute Unparseable Reward
# ============================================

def test_compute_unparseable_reward():
    """Unit test for the compute_unparseable_reward method."""
    print("🧪 Testing Exercise 3: Compute Unparseable Reward")
    print("="*50)
    
    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()
    
    # Test cases: (response, quality_dict, expected_reward_range, description)
    test_cases = [
        # Test 1: Long response with calculations (good attempt)
        {
            "response": "Let me work through this step by step. First I need to add the numbers together, then multiply by the factor mentioned. The calculation involves several steps that I'll show below.",
            "quality": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": False,
                "response_length": 180,
                "has_reasoning_words": True,
                "sentence_count": 3
            },
            "min_reward": 0.15,
            "max_reward": 0.25,
            "description": "Long response with calculations (>200 chars)"
        },
        
        # Test 2: Very short dismissive response
        {
            "response": "I don't know",
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": False,
                "response_length": 12,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.0,
            "max_reward": 0.0,
            "description": "Very short response (<20 chars)"
        },
        
        # Test 3: Medium response with some reasoning
        {
            "response": "First calculate the total, then find the average",
            "quality": {
                "has_calculation": False,
                "has_steps": True,
                "has_numbers": False,
                "response_length": 49,
                "has_reasoning_words": True,
                "sentence_count": 1
            },
            "min_reward": 0.05,
            "max_reward": 0.15,
            "description": "Medium response with reasoning"
        },
        
        # Test 4: Response with numbers but no final answer
        {
            "response": "We have 10 apples and 5 oranges to work with",
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 45,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.05,
            "max_reward": 0.15,
            "description": "Has numbers but no clear answer"
        },
        
        # Test 5: Long detailed attempt without parseable answer
        {
            "response": "To solve this problem, I would first identify all the given values. Then I'd set up the equation properly. Next, I'd solve for the unknown variable. Finally, I'd check my answer to make sure it makes sense in the context.",
            "quality": {
                "has_calculation": False,
                "has_steps": True,
                "has_numbers": False,
                "response_length": 225,
                "has_reasoning_words": True,
                "sentence_count": 4
            },
            "min_reward": 0.1,
            "max_reward": 0.25,
            "description": "Long detailed methodology without numbers"
        }
    ]
    
    passed = 0
    failed = 0
    
    for test_case in test_cases:
        response = test_case["response"]
        quality = test_case["quality"]
        min_reward = test_case["min_reward"]
        max_reward = test_case["max_reward"]
        description = test_case["description"]
        
        try:
            reward = test_model.compute_unparseable_reward(
                response=response,
                correct_answer=42.0,  # Arbitrary correct answer
                quality=quality,
                question="Test question"
            )
            
            if min_reward <= reward <= max_reward:
                print(f"✅ PASSED: {description}")
                print(f"   Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Response length: {quality['response_length']}")
                print(f"   Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                failed += 1
                
        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1
    
    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")
    
    if failed == 0:
        print("🎉 All tests passed!")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements:")
        print("- Long responses (>200 chars) with calculations → 0.2 reward")
        print("- Responses with reasoning and numbers → 0.1 reward")
        print("- Very short responses (<20 chars) → 0.0 reward")
        print("- Any reasonable attempt → at least 0.05 reward")
    
    return passed == len(test_cases)

# Run the test
test_compute_unparseable_reward()

🧪 Testing Exercise 3: Compute Unparseable Reward
✅ PASSED: Long response with calculations (>200 chars)
   Reward: 0.20 (expected: 0.15-0.25)
✅ PASSED: Very short response (<20 chars)
   Reward: 0.00 (expected: 0.00-0.00)
✅ PASSED: Medium response with reasoning
   Reward: 0.10 (expected: 0.05-0.15)
✅ PASSED: Has numbers but no clear answer
   Reward: 0.10 (expected: 0.05-0.15)
✅ PASSED: Long detailed methodology without numbers
   Reward: 0.20 (expected: 0.10-0.25)

Results: 5/5 passed, 0/5 failed
🎉 All tests passed!


True

In [12]:
### Reward High-Quality Correct Answers
def compute_correct_reward(self, response: str, predicted: float,
                            correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
    """
    Compute reward for correct answers.

    Base reward is 1.0, with bonuses for showing high-quality work.

    Args:
        response: The model's generated response
        predicted: The extracted numerical answer
        correct_answer: The correct numerical answer
        quality: Quality indicators from analyze_response_quality

    Returns:
        Reward between 1.0 and 1.3
    """

    reward = 1.0  # Base reward for correct answer

    if quality.get("has_steps",False):
        reward += 0.1
     # Add +0.1 bonus for detailed explanation (response_length > 100)
    if quality.get("response_length", False) > 100:
        reward += 0.1

    # Add +0.05 bonus for using proper reasoning words
    if quality.get("has_reasoning_words", False):
        reward += 0.05

    # Add +0.05 bonus for showing calculations
    if quality.get("has_calculation", False):
        reward += 0.05

    # Cap the maximum reward at 1.3
    reward = min(reward, 1.3)

    self.logger.info(f"✅ CORRECT ANSWER:")
    self.logger.info(f"  Predicted: {predicted}")
    self.logger.info(f"  Expected: {correct_answer}")
    self.logger.info(f"  Reward: {reward}")
    
    return reward

# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.compute_correct_reward = compute_correct_reward

In [13]:
# ============================================
# Compute Correct Reward
# ============================================

def test_compute_correct_reward():
    """Unit test for the compute_correct_reward method."""
    print("🧪 Testing Exercise 4: Compute Correct Reward")
    print("="*50)
    
    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()
    
    # Test cases with quality indicators and expected rewards
    # UPDATED: More strict test cases to properly test bonus implementation
    test_cases = [
        # Test 1: Perfect answer with all quality indicators - should get bonuses
        {
            "response": "Step 1: Add 5 + 3 = 8. Step 2: Multiply by 2 = 16. Therefore, the answer is 16.",
            "predicted": 16.0,
            "correct": 16.0,
            "quality": {
                "has_calculation": True,
                "has_steps": True,
                "has_reasoning": True,
                "response_length": 80,
                "sentence_count": 3
            },
            "min_reward": 1.15,  # Raised from 1.0 - must have bonuses for quality
            "max_reward": 1.3,
            "description": "Perfect answer with all quality indicators (should get bonuses)"
        },
        # Test 2: Correct but minimal response - should be exactly 1.0
        {
            "response": "16",
            "predicted": 16.0,
            "correct": 16.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_reasoning": False,
                "response_length": 2,
                "sentence_count": 1
            },
            "min_reward": 1.0,
            "max_reward": 1.0,  # Must be exactly 1.0 - no bonuses
            "description": "Correct but minimal response (no bonuses)"
        },
        # Test 3: Correct with some work but not perfect - should get partial bonuses
        {
            "response": "The calculation is 5 + 3 = 8, then 8 * 2 = 16.",
            "predicted": 16.0,
            "correct": 16.0,
            "quality": {
                "has_calculation": True,
                "has_steps": False,
                "has_reasoning": False,
                "response_length": 50,
                "sentence_count": 1
            },
            "min_reward": 1.05,  # Should get some bonus but not max
            "max_reward": 1.15,
            "description": "Correct with calculation but no steps (partial bonus)"
        },
    ]
    
    passed = 0
    failed = 0
    
    for test_case in test_cases:
        response = test_case["response"]
        predicted = test_case["predicted"]
        correct = test_case["correct"]
        quality = test_case["quality"]
        min_reward = test_case["min_reward"]
        max_reward = test_case["max_reward"]
        description = test_case["description"]
        
        try:
            reward = test_model.compute_correct_reward(
                response=response,
                predicted=predicted,
                correct_answer=correct,
                quality=quality,
            )
            
            if min_reward <= reward <= max_reward:
                print(f"✅ PASSED: {description}")
                print(f"   Reward: {reward:.3f} (expected: {min_reward:.3f} - {max_reward:.3f})")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Reward: {reward:.3f} (expected: {min_reward:.3f} - {max_reward:.3f})")
                failed += 1
                
        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1
    
    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")
    
    if failed == 0:
        print("🎉 All tests passed!")
    else:
        print("⚠️  Some tests failed. Please review your implementation:")
        print("Expected behavior:")
        print("- Base reward: 1.0 for correct answer")
        print("- +0.1 bonus for showing steps (has_steps=True)")
        print("- +0.1 bonus for detailed explanation (>100 chars)")
        print("- +0.05 bonus for using reasoning words")
        print("- +0.05 bonus for showing calculations")
        print("- Max reward: 1.3")
    
    return passed == len(test_cases)

# Run the test
test_compute_correct_reward()

🧪 Testing Exercise 4: Compute Correct Reward
✅ PASSED: Perfect answer with all quality indicators (should get bonuses)
   Reward: 1.150 (expected: 1.150 - 1.300)
✅ PASSED: Correct but minimal response (no bonuses)
   Reward: 1.000 (expected: 1.000 - 1.000)
✅ PASSED: Correct with calculation but no steps (partial bonus)
   Reward: 1.050 (expected: 1.050 - 1.150)

Results: 3/3 passed, 0/3 failed
🎉 All tests passed!


True

In [14]:
### Implement Partial Credit for Wrong Answers
def compute_wrong_reward(self, response: str, predicted: float,
                        correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
    """
    Compute partial credit for wrong answers.

    This is critical for learning! Instead of 0.0 for all wrong answers,
    give partial credit based on:
    1. How close the answer is
    2. Whether work was shown
    3. Quality of reasoning

    Args:
        response: The model's generated response
        predicted: The extracted numerical answer
        correct_answer: The correct numerical answer
        quality: Quality indicators from analyze_response_quality

    Returns:
        Reward between 0.1 and 0.9
    """

    reward = 0.0  # Currently no partial credit - this is the main problem!

    # calculate relative error -  normalized error  
    if correct_answer != 0:
        relative_error = abs(predicted - correct_answer) / abs(correct_answer)
    else:
        relative_error = abs(predicted - correct_answer) # when false answer 

    if relative_error < 0.01: # 1% error 
        reward = 0.9
    elif relative_error <= 0.10: # 10% error
        reward = 0.3
    elif relative_error <= 0.30: # 30% error 
        reward = 0.1
    elif (0.1 <= (predicted / correct_answer) <=10 ) and (relative_error > 0.5) if correct_answer !=0 else False: # between 0.1x and 10x for large erros -> 0.3 reward 
        reward = 0.3
    else:
        reward = 0.1

    # Add bonus (+0.1) for showing calculations and steps (has_calculation and has_steps)
    if quality.get("has_calculation", False) and quality.get("has_steps", False):
        reward += 0.1
    
    # Ensure minimum reward for any attempt
    reward = max(0.1, reward)

    # Cap at 0.9 (to keep it below correct answers)
    reward = min(0.9, reward)

    self.logger.info(f"❌ WRONG ANSWER:")
    self.logger.info(f"  Predicted: {predicted}")
    self.logger.info(f"  Expected: {correct_answer}")
    self.logger.info(f"  Reward: {reward}")

    return reward

# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.compute_wrong_reward = compute_wrong_reward

In [15]:
# ============================================
# Compute Wrong Reward
# ============================================

def test_compute_wrong_reward():
    """Unit test for the compute_wrong_reward method."""
    print("🧪 Testing Exercise 5: Compute Wrong Reward")
    print("="*50)
    
    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()
    
    # Test cases with various error levels
    test_cases = [
        # Test 1: Almost correct (within 1%)
        {
            "response": "The answer is 100.5",
            "predicted": 100.5,
            "correct": 100.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 19,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.85,
            "max_reward": 0.95,
            "description": "Within 1% error (0.5% off)"
        },
        
        # Test 2: Close but wrong (within 10%)
        {
            "response": "After calculations, I get 45",
            "predicted": 45.0,
            "correct": 50.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 28,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.3,
            "max_reward": 0.4,
            "description": "Within 10% error"
        },
        
        # Test 3: Moderately wrong (within 30%)
        {
            "response": "The result is 70",
            "predicted": 70.0,
            "correct": 100.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 16,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.1,  # 30% error (0.3) falls into >30% bracket (0.1 base)
            "max_reward": 0.15,
            "description": "Within 30% error"
        },
        
        # Test 4: Right order of magnitude
        {
            "response": "Approximately 500",
            "predicted": 500.0,
            "correct": 100.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 17,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.1,
            "max_reward": 0.35,
            "description": "Right order of magnitude (5x off)"
        },
        
        # Test 5: Wrong but shows work
        {
            "response": "Step 1: Add 10 + 20 = 30. Step 2: Multiply by 3 = 90. The answer is 90.",
            "predicted": 90.0,
            "correct": 60.0,
            "quality": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": True,
                "response_length": 72,
                "has_reasoning_words": False,
                "sentence_count": 3
            },
            "min_reward": 0.15,  # 0.5 for 50% error + 0.1 for showing work
            "max_reward": 0.25,
            "description": "Wrong but shows detailed work"
        },
        
        # Test 6: Very wrong
        {
            "response": "The answer is 1000000",
            "predicted": 1000000.0,
            "correct": 10.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 21,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.05,
            "max_reward": 0.15,
            "description": "Very wrong answer"
        }
    ]
    
    passed = 0
    failed = 0
    
    for test_case in test_cases:
        response = test_case["response"]
        predicted = test_case["predicted"]
        correct = test_case["correct"]
        quality = test_case["quality"]
        min_reward = test_case["min_reward"]
        max_reward = test_case["max_reward"]
        description = test_case["description"]
        
        try:
            reward = test_model.compute_wrong_reward(
                response=response,
                predicted=predicted,
                correct_answer=correct,
                quality=quality
            )
            
            # Calculate actual error for debugging
            relative_error = abs(predicted - correct) / (abs(correct) + 1e-10)
            
            if min_reward <= reward <= max_reward:
                print(f"✅ PASSED: {description}")
                print(f"   Error: {relative_error:.2%}, Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Predicted: {predicted}, Correct: {correct}, Error: {relative_error:.2%}")
                print(f"   Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                failed += 1
                
        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1
    
    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")
    
    if failed == 0:
        print("🎉 All tests passed!")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements (partial credit system):")
        print("- Within 1% error → 0.9 reward")
        print("- Within 10% error → 0.7 reward")
        print("- Within 30% error → 0.5 reward")
        print("- Right order of magnitude (0.1x-10x) → 0.3 reward")
        print("- Any reasonable attempt → 0.1 reward")
        print("- +0.1 bonus for showing work (calculations + steps)")
    
    return passed == len(test_cases)

# Run the test
test_compute_wrong_reward()

🧪 Testing Exercise 5: Compute Wrong Reward
✅ PASSED: Within 1% error (0.5% off)
   Error: 0.50%, Reward: 0.90 (expected: 0.85-0.95)
✅ PASSED: Within 10% error
   Error: 10.00%, Reward: 0.30 (expected: 0.30-0.40)
✅ PASSED: Within 30% error
   Error: 30.00%, Reward: 0.10 (expected: 0.10-0.15)
✅ PASSED: Right order of magnitude (5x off)
   Error: 400.00%, Reward: 0.30 (expected: 0.10-0.35)
✅ PASSED: Wrong but shows detailed work
   Error: 50.00%, Reward: 0.20 (expected: 0.15-0.25)
✅ PASSED: Very wrong answer
   Error: 9999900.00%, Reward: 0.10 (expected: 0.05-0.15)

Results: 6/6 passed, 0/6 failed
🎉 All tests passed!


True

In [16]:
# compute_reward method (main orchestrator)\
def compute_reward(self, response: str, correct_answer: float, question: str = None) -> float:
    """
    Main reward computation function - delegates to specialized methods.
    
    This function orchestrates the reward computation by:
    1. Extracting numerical answer from response
    2. Analyzing response quality indicators  
    3. Calling appropriate reward computation method
    4. Returning final reward value
    
    Args:
        response: The model's response text
        correct_answer: The correct numerical answer
        question: The original question (optional)
        
    Returns:
        float: Final reward value for this response
    """
    # extract numerical answer 
    predicted = self.extract_numerical_answer(response)
    # analyze response quality
    quality = self.analyze_response_quality(response)
    # route to appropiate reward computation 
    if predicted is None:
        # could not parse any numerical answer 
        return self.compute_unparseable_reward(response,correct_answer,quality,question)
    elif abs(predicted - correct_answer) < 0.01:
        # answer is correct with small tolerance 
        return self.compute_correct_reward(response,predicted,correct_answer,quality,question)
    else :
        # answer is wrong 
        return self.compute_wrong_reward(response,predicted,correct_answer,quality,question)

# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.compute_reward = compute_reward

print("✅ Reward model class defined!")
print("This will be your 'grading system' for model responses.")

✅ Reward model class defined!
This will be your 'grading system' for model responses.


In [17]:
# ============================================
# Test the Reward Model
# ============================================

# Test your reward model with different types of answers
print("🧪 TESTING THE REWARD MODEL")
print("="*60)
print("See how it grades different types of responses:\n")

reward_model = GSM8KRewardSignal()
print("✅ Reward model created with detailed logging for debugging!")
print("Check the log file to see question, response, and reward details for each generation.")

# Test cases: (response, correct_answer)
test_cases = [
    # Perfect answer with steps
    ("Let me calculate step by step: 5 + 3 = 8. The answer is #### 8", 8.0),
    
    # Correct but no steps
    ("The answer is 8", 8.0),
    
    # Close but wrong
    ("5 + 3 = 7. So the answer is 7.", 8.0),
    
    # Very wrong
    ("The total is 100", 8.0),
    
    # Shows work but wrong
    ("First I add 5 + 3 = 9. Then I multiply by 2 = 18. Answer: 18", 8.0),
    
    # No attempt
    ("I don't know how to solve this", 8.0),
]

for i, (response, correct) in enumerate(test_cases, 1):
    print(f"Test {i}:")
    print(f"  Response: \"{response[:50]}...\"" if len(response) > 50 else f"  Response: \"{response}\"")
    
    # Compute reward
    reward = reward_model.compute_reward(response, correct, "Test question")
    
    # Extract answer for display
    extracted = reward_model.extract_numerical_answer(response)
    
    print(f"  Extracted answer: {extracted}")
    print(f"  Correct answer: {correct}")
    print(f"  Reward: {reward:.2f}")
    
    # Explain the reward
    if reward >= 1.0:
        print("  ✅ Excellent!")
    elif reward >= 0.5:
        print("  🟨 Good attempt")
    elif reward >= 0.2:
        print("  🟠 Some credit")
    else:
        print("  ⚪ Minimal credit")
    print()


🧪 TESTING THE REWARD MODEL
See how it grades different types of responses:

✅ Reward model created with detailed logging for debugging!
Check the log file to see question, response, and reward details for each generation.
Test 1:
  Response: "Let me calculate step by step: 5 + 3 = 8. The answ..."
  Extracted answer: 8.0
  Correct answer: 8.0
  Reward: 1.15
  ✅ Excellent!

Test 2:
  Response: "The answer is 8"
  Extracted answer: 8.0
  Correct answer: 8.0
  Reward: 1.00
  ✅ Excellent!

Test 3:
  Response: "5 + 3 = 7. So the answer is 7."
  Extracted answer: 7.0
  Correct answer: 8.0
  Reward: 0.20
  🟠 Some credit

Test 4:
  Response: "The total is 100"
  Extracted answer: 100.0
  Correct answer: 8.0
  Reward: 0.10
  ⚪ Minimal credit

Test 5:
  Response: "First I add 5 + 3 = 9. Then I multiply by 2 = 18. ..."
  Extracted answer: 18.0
  Correct answer: 8.0
  Reward: 0.40
  🟠 Some credit

Test 6:
  Response: "I don't know how to solve this"
  Extracted answer: None
  Correct answer: 8.0
  

## load the language model 

In [26]:
# ============================================
# Load the Tokenizer
# ============================================

print("Loading tokenizer...")
print("The tokenizer converts text to tokens (numbers) that the model understands.\n")

is_local = os.path.exists(config.model_name)

# Load the tokenizer
# trust_remote_code=True allows loading custom code from the model repository
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,  # Some models have custom tokenizer code
    local_files_only=is_local,
    enable_thinking=False
)

# Set up padding token
# Padding is used to make all inputs the same length
if tokenizer.pad_token is None:
    # If no padding token defined, use the end-of-sequence token
    tokenizer.pad_token = tokenizer.eos_token

# Set padding to left side for generation tasks
# This ensures the actual text is on the right (where model expects it)
tokenizer.padding_side = "left"

print("✅ Tokenizer loaded successfully!")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")
print(f"  Padding token: {tokenizer.pad_token}")
print(f"  End-of-sequence token: {tokenizer.eos_token}")

# Example of tokenization
example_text = "What is 5 + 3?"
tokens = tokenizer.encode(example_text)
print(f"\nExample tokenization:")
print(f"  Text: \"{example_text}\"")
print(f"  Tokens: {tokens[:10]}..." if len(tokens) > 10 else f"  Tokens: {tokens}")
print(f"  Number of tokens: {len(tokens)}")

Loading tokenizer...
The tokenizer converts text to tokens (numbers) that the model understands.

✅ Tokenizer loaded successfully!
  Vocabulary size: 151,669 tokens
  Padding token: <|endoftext|>
  End-of-sequence token: <|im_end|>

Example tokenization:
  Text: "What is 5 + 3?"
  Tokens: [3838, 374, 220, 20, 488, 220, 18, 30]
  Number of tokens: 8


In [ ]:
# ============================================
# Load the Language Model
# ============================================

print("Loading the Qwen3 0.6B model...")
print("This may take 1-2 minutes depending on your internet speed.\n")

# Set up model loading arguments
model_kwargs = {
    "trust_remote_code": True,  # Allow custom model code
    "dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    "use_cache": True,
    "local_files_only": is_local,
    # bfloat16: Uses less memory than float32 but maintains good precision
    # float32: Full precision (used on CPU)
}

# Optional: Use 8-bit quantization to save memory
if config.use_8bit:
    print("Using 8-bit mode to save memory...")
    model_kwargs["load_in_8bit"] = True
    # 8-bit mode reduces memory usage by ~50% with minimal accuracy loss

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    attn_implementation="sdpa",  # Native PyTorch fused attention
    torch_dtype=torch.float16,   # Match fp16
    **model_kwargs
)
model.to(device)
print("✅ Model loaded successfully!")

# Display model information
print(f"\nModel Information:")
print(f"  Model type: {model.__class__.__name__}")
print(f"  Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion")

# Check model device
if next(model.parameters()).is_cuda:
    print(f"  Location: GPU (fast training!)")
else:
    print(f"  Location: CPU (slower training)")

In [ ]:
# ============================================
# Enable Memory Optimizations
# ============================================

print("Enabling memory optimizations...\n")

# Enable gradient checkpointing 
# this trades computation for memory by not storing all intermediate values 
if hasattr(model,'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")
    print("   This saves memory by recomputing values when needed")
    print("   Training will be slightly slower but use less memory")
else:
    print("Gradient checkpointing not available for this model")

# Test the model with a simple generation
print("\nTesting model with a simple math problem...")
test_prompt = "What is 2 + 2? The answer is"
inputs = tokenizer(test_prompt, return_tensors="pt")

# Move inputs to same device as model
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

# Generate a short response
with torch.no_grad():  # Don't calculate gradients for this test
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.1,  # Low temperature for deterministic output
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id  # Explicitly set to suppress warning
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"  Prompt: \"{test_prompt}\"")
print(f"  Model response: \"{response}\"")
print("\n✅ Model is working correctly!")

### Prepare Training and Validation Datasets

In [25]:
import re
from typing import Tuple
from datasets import Dataset, load_dataset


def prepare_dataset(config, tokenizer) -> Tuple[Dataset, Dataset]:
    """Loads GSM8K training set, splits it into train/validation,

    and formats each problem with the prompt template and numerical answer.
    """
    # 1. Load the GSM8K dataset (use 'openai/gsm8k' rather than standalone 'gsm8k')
    try:
        from datasets import load_from_disk
        raw_data = load_from_disk("/app/data/gsm8k")["train"]
    except Exception:
        raw_data = load_dataset("openai/gsm8k", "main", split="train")

    # 2. Split into train and validation sets
    train_ratio = getattr(config, "train_split_ratio", 0.8)
    seed = getattr(config, "seed", 42)
    split_dataset = raw_data.train_test_split(
        train_size=train_ratio, seed=seed, shuffle=True
    )
    train_data = split_dataset["train"]
    eval_data = split_dataset["test"]

    # 3. Helper to extract the numerical answer from the ground-truth text
    def extract_answer(answer_text: str) -> float:
        if "####" in answer_text:
            ans_str = (
                answer_text.split("####")[-1]
                .strip()
                .replace(",", "")
                .replace("$", "")
            )
            try:
                return float(ans_str)
            except ValueError:
                pass
        numbers = re.findall(r"[+-]?\d+(?:\.\d+)?", answer_text.replace(",", ""))
        return float(numbers[-1]) if numbers else 0.0

    # 4. Formatting function matching the lab template
    def format_example(example):
        question = example["question"].strip()
        answer_text = example["answer"]

        prompt = (
            f"Question: {question}\n\n"
            f"Let's solve this step-by-step and find the numerical answer:\n"
        )

        return {
            "prompt": prompt,
            "question": question,
            "answer": extract_answer(answer_text),
            "answer_text": answer_text,
        }

    # 5. Map formatting across train and evaluation sets
    train_dataset = train_data.map(
        format_example, desc="Formatting train dataset"
    )
    eval_dataset = eval_data.map(format_example, desc="Formatting eval dataset")

    return train_dataset, eval_dataset

In [27]:
# ============================================
# Prepare Training and Validation Datasets
# ============================================

print("Preparing datasets for training...\n")

# Use the utility function to prepare the datasets
# This function:
# 1. Loads the GSM8K training data
# 2. Splits it into train/validation
# 3. Formats each problem with the prompt template
# 4. Extracts numerical answers
train_dataset, eval_dataset = prepare_dataset(config, tokenizer)

# Display dataset statistics
print("\nDataset Statistics:")
print(f"  Total examples: {len(train_dataset) + len(eval_dataset):,}")
print(f"  Training examples: {len(train_dataset):,} ({len(train_dataset)/(len(train_dataset)+len(eval_dataset))*100:.1f}%)")
print(f"  Validation examples: {len(eval_dataset):,} ({len(eval_dataset)/(len(train_dataset)+len(eval_dataset))*100:.1f}%)")

# Calculate training iterations
steps_per_epoch = len(train_dataset) // (config.per_device_train_batch_size * config.gradient_accumulation_steps)
total_steps = steps_per_epoch * config.num_train_epochs
print(f"\nTraining Iterations:")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {total_steps}")
print(f"  Evaluations during training: {total_steps // config.eval_steps}")

Preparing datasets for training...


Dataset Statistics:
  Total examples: 7,473
  Training examples: 5,978 (80.0%)
  Validation examples: 1,495 (20.0%)

Training Iterations:
  Steps per epoch: 93
  Total training steps: 465
  Evaluations during training: 23


In [28]:
# ============================================
# Show Sample Training Examples
# ============================================

print("📝 Sample Training Examples:")
print("="*60)

# Show 3 examples from the training set
for i in range(min(3, len(train_dataset))):
    sample = train_dataset[i]
    
    print(f"\nExample {i+1}:")
    print("-"*40)
    
    # Show the prompt (that you give the model)
    print("PROMPT (Input to model):")
    print(sample['prompt'][:300] + "..." if len(sample['prompt']) > 300 else sample['prompt'])
    
    # Show the expected answer
    print(f"\nEXPECTED NUMERICAL ANSWER: {sample['answer']}")
    
    # Show part of the solution
    if 'answer_text' in sample:
        print("\nSOLUTION STEPS (first part):")
        print(sample['answer_text'][:200] + "..." if len(sample['answer_text']) > 200 else sample['answer_text'])

print("\n" + "="*60)
print("The model will learn to generate step-by-step solutions like these!")

📝 Sample Training Examples:

Example 1:
----------------------------------------
PROMPT (Input to model):
Question: Alicia had a wonderful birthday party where she got lots of presents. 10 of the presents were in small boxes. 12 of the presents were in medium boxes. A third of all the presents she is given are in large boxes. How many presents did Alicia get for her birthday?

Let's solve this step-by-s...

EXPECTED NUMERICAL ANSWER: 33.0

SOLUTION STEPS (first part):
Alicia got 10 small boxed presents + 12 medium boxed presents = <<10+12=22>>22 presents.
A third of all the presents Alicia gets are in large boxes, 22 / 2 = <<22/2=11>>11 is half of the presents we k...

Example 2:
----------------------------------------
PROMPT (Input to model):
Question: A jar on the family's counter contains change they've been saving a trip to the ice cream shop. There are 123 pennies, 85 nickels, 35 dimes, and a number of quarters. All five family members get a double scoop, which costs $3 each. Aft

### Create Evaluation Callback

In [20]:
import re
from typing import List, Optional
import torch
from transformers import TrainerCallback


class GSM8KEvaluationCallback(TrainerCallback):
    """Evaluation callback that periodically evaluates model accuracy and

    generation quality on a test dataset during GRPO training.
    """

    def __init__(
        self,
        tokenizer,
        test_dataset,
        batch_size: int = 64,
        sample_size: float = 1.0,
        reward_model=None,
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.test_dataset = test_dataset
        self.batch_size = batch_size
        self.sample_size = sample_size
        self.reward_model = reward_model

        # Subsample test dataset if sample_size < 1.0
        if 0.0 < self.sample_size < 1.0:
            num_samples = int(len(self.test_dataset) * self.sample_size)
            self.eval_dataset = self.test_dataset.select(range(num_samples))
        else:
            self.eval_dataset = self.test_dataset

    @staticmethod
    def extract_answer(text: str) -> Optional[float]:
        """Extracts numerical answer from model output."""
        if "####" in text:
            ans = (
                text.split("####")[-1].strip().replace(",", "").replace("$", "")
            )
            try:
                return float(ans)
            except ValueError:
                pass

        patterns = [
            r"(?:The answer is|answer:|Answer:)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"(?:equals?|=)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"(?:total|sum|result|Total|Final answer)\s*(?:is|:|=)?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"Therefore,?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
            if matches:
                try:
                    return float(
                        matches[-1].replace("$", "").replace(",", "")
                    )
                except ValueError:
                    continue

        numbers = re.findall(r"[+-]?\d+(?:\.\d+)?", text.replace(",", ""))
        if numbers:
            try:
                return float(numbers[-1])
            except ValueError:
                pass
        return None

    def on_step_end(self, args, state, control, **kwargs):
        """Runs evaluation every `eval_steps`."""
        if state.global_step == 0 or state.global_step % args.eval_steps != 0:
            return control

        model = kwargs.get("model")
        if model is None:
            return control

        model.eval()
        correct = 0
        total = 0
        total_reward = 0.0
        device = next(model.parameters()).device

        print(
            f"\n--- Running Evaluation at Step {state.global_step} ({len(self.eval_dataset)} samples) ---"
        )

        with torch.no_grad():
            for i in range(0, len(self.eval_dataset), self.batch_size):
                batch = self.eval_dataset[i : i + self.batch_size]
                questions = batch["question"]
                answers = batch["answer"]

                prompts = [
                    f"Question: {q.strip()}\n\nLet's solve this step-by-step and find the numerical answer:\n"
                    for q in questions
                ]

                # Tokenize inputs with left padding
                inputs = self.tokenizer(
                    prompts, return_tensors="pt", padding=True, truncation=True
                ).to(device)

                # Generate completions
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=400,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

                # Extract only the newly generated tokens
                prompt_len = inputs["input_ids"].shape[1]
                generated_tokens = outputs[:, prompt_len:]
                completions = self.tokenizer.batch_decode(
                    generated_tokens, skip_special_tokens=True
                )

                for comp, ground_truth, q in zip(completions, answers, questions):
                    pred = self.extract_answer(comp)

                    # Extract ground truth float
                    if isinstance(ground_truth, (int, float)):
                        gt_val = float(ground_truth)
                    else:
                        gt_val = self.extract_answer(str(ground_truth))

                    # Accuracy check
                    is_correct = (
                        pred is not None
                        and gt_val is not None
                        and abs(pred - gt_val) < 0.01
                    )
                    if is_correct:
                        correct += 1
                    total += 1

                    # Reward score check if reward_model is supplied
                    if self.reward_model is not None and gt_val is not None:
                        total_reward += self.reward_model.compute_reward(
                            comp, gt_val, q
                        )

        accuracy = (correct / total) * 100 if total > 0 else 0.0
        avg_reward = total_reward / total if total > 0 else 0.0

        print(
            f"Step {state.global_step} Evaluation: Accuracy = {accuracy:.2f}% ({correct}/{total})"
        )
        if self.reward_model is not None:
            print(f"Step {state.global_step} Evaluation: Avg Reward = {avg_reward:.3f}")

        model.train()
        return control

In [29]:
## this evalutaion matters 
# what it tracks ?
## is model getting better? , what the current accuracy? , should your stop the traiing ?
# evaluation callbacks measure (accuracy , average reward, sample outputs)
# evalutation frequency (every 20 training steps, on test set, using sample for seed)

# ============================================
# Set Up Evaluation System
# ============================================

print("Setting up evaluation system...\n")

# Load the test dataset for evaluation
# IMPORTANT: you should use the TEST set (not training data) to measure true performance
print("Loading GSM8K test dataset...")
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
test_dataset = gsm8k
print(f"✅ Loaded {len(test_dataset):,} test examples")
print("   These are NEVER seen during training (prevents cheating)\n")

# Create the evaluation callback
# This will run periodically during training to check progress
eval_callback = GSM8KEvaluationCallback(
    tokenizer=tokenizer,
    test_dataset=test_dataset,
    batch_size=64,  # How many examples to evaluate at once
    sample_size=1.0  # Use full test set (1.0 = 100%)
)

print("✅ Evaluation system configured!")
print(f"\nEvaluation Details:")
print(f"  Will evaluate on: {len(test_dataset)} test examples")
print(f"  Evaluation frequency: Every {config.eval_steps} training steps")
print(f"  Metrics tracked: Accuracy, Average Reward, Sample Outputs")
print(f"\nTip: Watch the accuracy increase during training!")

Setting up evaluation system...

Loading GSM8K test dataset...
✅ Loaded 1,319 test examples
   These are NEVER seen during training (prevents cheating)

✅ Evaluation system configured!

Evaluation Details:
  Will evaluate on: 1319 test examples
  Evaluation frequency: Every 20 training steps
  Metrics tracked: Accuracy, Average Reward, Sample Outputs

Tip: Watch the accuracy increase during training!


### Configure grpo trainer 

##### Here's how GRPO training works:

* ***Generate Multiple Answers:*** For each question, generate 12 different answers
* ***Score Each Answer:***  Use your reward model to grade each one
* ***Compare Within Group:***  See which answers are better than others
* ***Update Model:*** Teach it to prefer the better answers
Instead of saying "this answer is worth 0.7 points", GRPO says "Answer #3 is better than answers #1, #2, #4, #5...". This relative comparison is more stable and effective than scoring each answer

Key Components
* ### Trainer: Orchestrates the training process
* ### Reward Function: Grades the generated answers
* ### Configuration: All your settings from earlier
Note the GRPOConfig() within the create_grpo_trainer() function. This is where you set various parameters such as the temperature, as you have seen in the video.

In [22]:
 def create_grpo_trainer(config, model, tokenizer, train_dataset, eval_dataset, reward_model, test_dataset=None):
    """
    Create and configure the GRPO trainer.
    
    This function sets up everything needed for GRPO training:
    1. Configuration for the training process
    2. Reward computation function
    3. The trainer itself
    
    Args:
        config: Training configuration
        model: The language model to train
        tokenizer: Tokenizer for the model
        train_dataset: Training data
        eval_dataset: Validation data
        reward_model: Your reward scoring system
        test_dataset: Test data for evaluation
    
    Returns:
        A configured GRPO trainer ready to train
    """
    
    print("🔧 Creating GRPO trainer...\n")
    prompt2ans = {}
    for item in train_dataset:
        prompt2ans[item["prompt"]] = (item["answer"],item["question"])
    print(f"Built answer lookup with {len(prompt2ans):,} entries")

    # configure the grpo trainer parameters 
    GRPO_config = GRPOConfig(
        num_generations = config.num_generations, # generate 12 answer per questions
        temperature = config.temperature,
        generation_kwargs = {
            "max_new_tokens": config.max_new_tokens,
            "temperature": config.temperature,
            "top_p": 0.95,  # Only consider top 95% probability tokens
            "do_sample": True,  # Enable random sampling
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        },
        generation_batch_size = min(96,config.per_device_train_batch_size * config.num_generations),
        output_dir = config.output_dir,
        num_train_epochs = config.num_train_epochs,
        
        per_device_train_batch_size=config.per_device_train_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        
        learning_rate=config.learning_rate,
        lr_scheduler_type="cosine",  # Gradually reduce learning rate
        warmup_steps=0.1,  # Start with lower learning rate for stability

        max_grad_norm=1.0,  # Clip gradients to prevent explosions
        adam_beta1=0.9,  # Momentum parameter
        adam_beta2=0.95,  # Better for RL than default 0.999
        weight_decay=0.1,  # L2 regularization to prevent overfitting

        logging_steps=config.logging_steps,
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        save_total_limit=config.save_total_limit,

        seed=config.seed,
        bf16 = False,
        fp16 = True, # JUST BECAUSE T4 2X
        remove_unused_columns=False,
        push_to_hub=False,  # Don't upload to Hugging Face
        report_to=[],  # Disable external logging

        # use_vllm = True,
        # vllm_gpu_memory_utilization = 0.3,
    )
    print("✅ GRPO configuration created")
    return GRPO_config, prompt2ans

# Create the configuration
GRPO_config, prompt2ans = create_grpo_trainer(config, model, tokenizer, train_dataset, eval_dataset, reward_model, test_dataset)
print("\nConfiguration ready for training!")

NameError: name 'model' is not defined

### Reward compuation function defined with detailed logging
This function will:

* Score each generated answer with detailed logs
* Show question, response, and reward for each generation
* Compare answers within groups
* Return relative rewards for training
* Log group statistics for debugging

In this function you can find the implementation of both the mean_reward and normalized_rewards, which you saw in the video.

In [30]:
def compute_rewards(prompts: List[str], completions: List[str], **kwargs):
    """
    Compute rewards for generated completions with detailed logging.
    
    This is the heart of GRPO - it scores each generated answer
    and normalizes within groups for relative comparison.
    
    Process:
    1. Group completions by their prompt
    2. Score each completion with detailed logging
    3. Normalize scores within each group
    4. Return normalized rewards
    
    Args:
        prompts: List of prompts (with duplicates for each generation)
        completions: List of generated responses
    
    Returns:
        List of normalized rewards
    """
    
    rewards = []

    # Log batch information
    logger.info(f"\n{'='*80}")
    logger.info(f"GRPO REWARD COMPUTATION")
    logger.info(f"Total completions: {len(completions)}")
    logger.info(f"Total prompts: {len(prompts)}")
    logger.info(f"Expected generations per prompt: {config.num_generations}")
    logger.info(f"{'='*80}")

    # get uniqu prompts (remove duplicate)
    seen = set()
    unique_prompt = []
    for p in prompts:
        if p not in seen:
            unique_prompt.append(p)
            seen.add(p)
    num_unique_prompts = len(unique_prompt)
    logger.info(f"Calculated unique prompts: {num_unique_prompts}")
    
    # Debug: Print samples
    for i in range(min(4, len(completions))):
        logger.info(f"Completion {i+1}: {completions[i][:80]}...")

    # Process each unique prompt and its completions
    for i, unique_prompt in enumerate(unique_prompts):
        # Find all completions for this prompt
        prompt_indices = [idx for idx, p in enumerate(prompts) if p == unique_prompt]
        group_completions = [completions[idx] for idx in prompt_indices]
        
        # Get correct answer from your lookup dictionary
        correct_answer, question = prompt2ans.get(unique_prompt, (0.0, "Unknown question"))
        
        logger.info(f"\n--- Unique Prompt {i+1}/{num_unique_prompts} ---")
        logger.info(f"Question: {question[:150]}...")
        logger.info(f"Expected Answer: {correct_answer}")
        logger.info(f"Number of generations for this prompt: {len(group_completions)}")
        
        # Compute reward for each completion with detailed logging
        group_rewards = []
        for j, completion in enumerate(group_completions):
            logger.info(f"\n  Generation {j+1}/{len(group_completions)}:")
            reward = reward_model.compute_reward(completion, correct_answer, question)
            group_rewards.append(reward)
        
        # Normalize rewards within the group (mean-centering)
        # This makes rewards relative: positive = better than average, negative = worse
        if len(group_rewards) > 1:
            mean_reward = sum(group_rewards) / len(group_rewards)
            # Mean-center only - no standard deviation scaling
            normalized_rewards = [r - mean_reward for r in group_rewards]
        else:
            # Single generation - no comparison possible
            normalized_rewards = [0.0]
        
        # Add normalized rewards in correct order
        for idx, norm_reward in zip(prompt_indices, normalized_rewards):
            if len(rewards) <= idx:
                rewards.extend([0] * (idx - len(rewards) + 1))
            rewards[idx] = norm_reward
        
        # Log group statistics matching your format
        avg_reward = sum(group_rewards) / len(group_rewards)
        max_reward = max(group_rewards)
        min_reward = min(group_rewards)
        avg_norm = sum(normalized_rewards) / len(normalized_rewards)
        logger.info(f"\n  Group Summary: Avg={avg_reward:.3f}, Max={max_reward:.3f}, Min={min_reward:.3f}")
        logger.info(f"  Normalized: Avg={avg_norm:.3f}, Rewards={[f'{r:.3f}' for r in normalized_rewards]}")
    
    overall_avg = sum(rewards) / len(rewards) if rewards else 0
    logger.info(f"\n{'='*80}")
    logger.info(f"BATCH SUMMARY: Overall Average Reward = {overall_avg:.3f}")
    logger.info(f"{'='*80}\n")
    
    return rewards

In [ ]:
# ## create the grpo trainer 
# print("Creating the GRPO trainer...\n")

# # Initialize the GRPO trainer with all the components
# trainer = GRPOTrainer(
#     model=model,  # The language model to train
#     reward_funcs=compute_rewards,  # Function to compute rewards
#     args=GRPO_config,  # Training configuration
#     train_dataset=train_dataset,  # Training data
#     eval_dataset=eval_dataset,  # Validation data
#     processing_class=tokenizer,  # Tokenizer
# )

# # Add the evaluation callback
# trainer.add_callback(eval_callback)
# print("✅ Added evaluation callback for progress tracking")

# print("\n✅ GRPO Trainer created successfully!")
# print("\nTraining Summary:")
# print(f"  Model: DeepSeek Math 7B")
# print(f"  Dataset: GSM8K math problems")
# print(f"  Training examples: {len(train_dataset):,}")
# print(f"  Generations per prompt: {config.num_generations}")
# print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
# print(f"  Total training steps: ~{len(train_dataset) // (config.per_device_train_batch_size * config.gradient_accumulation_steps) * config.num_train_epochs}")
# print(f"\nThe model will now learn to solve math problems better!")

In [ ]:
! pip install accelerate 

In [35]:
# 1. Build prompt-to-answer lookup
prompt2ans = {}
for item in train_dataset:
    prompt2ans[item["prompt"]] = (item["answer"], item["question"])

# 1. Build prompt-to-answer lookup
prompt2ans = {}
for item in train_dataset:
    prompt2ans[item["prompt"]] = (item["answer"], item["question"])

# 2. Define GRPOConfig (Config only — does not initialize CUDA or Accelerator)
GRPO_config = GRPOConfig(
    output_dir = config.output_dir,
    num_train_epochs = config.num_train_epochs,
    per_device_train_batch_size = config.per_device_train_batch_size,
    gradient_accumulation_steps = config.gradient_accumulation_steps,
    generation_batch_size = 24,
    learning_rate = config.learning_rate,
    lr_scheduler_type = "cosine",
    warmup_steps = 0.1,
    max_grad_norm = 1.0,
    num_generations = config.num_generations,
    temperature = config.temperature,
    logging_steps = config.logging_steps,
    eval_steps = config.eval_steps,
    save_steps = config.save_steps,
    save_total_limit = config.save_total_limit,
    seed = config.seed,
    fp16 = True,
    bf16 = False,
    remove_unused_columns = False,
    push_to_hub = False,
    report_to = [],
)

In [36]:
from accelerate import notebook_launcher
from trl import GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

def train_fn():
    # Load model and tokenizer INSIDE the worker process
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name, 
        trust_remote_code=True, 
        enable_thinking=False
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        attn_implementation="sdpa",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=compute_rewards,
        args=GRPO_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )
    
    trainer.train()

notebook_launcher(train_fn, num_processes=2)

Launching training on 2 CUDAs.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835] failed (exitcode: 1) local_rank: 0 (pid: 483) of fn: train_fn (start_method: fork)
E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835] Traceback (most recent call last):
E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835]   File "/usr/local/lib/python3.13/dist-packages/torch/distributed/elastic/multiprocessing/api.py", line 790, in _poll
E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835]     self._pc.join(-1)
E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835]     ~~~~~~~~~~~~~^^^^
E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835]   File "/usr/local/lib/python3.13/dist-packages/torch/multiprocessing/spawn.py", line 211, in join
E1027 07:47:53.898000 234 torch/distributed/elastic/multiprocessing/api.py:835]     raise ProcessRaisedException(msg, error_index, failed_pro

ChildFailedError: 
============================================================
train_fn FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-10-27_07:47:53
  host      : 2b165d3a38fe
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 483)
  error_file: /tmp/torchelastic_afiik2_d/none_zyg3igbu/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/usr/local/lib/python3.13/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
      return f(*args, **kwargs)
    File "/tmp/ipykernel_234/243589684.py", line 24, in train_fn
      trainer = GRPOTrainer(
          model=model,
      ...<4 lines>...
          processing_class=tokenizer,
      )
    File "/usr/local/lib/python3.13/dist-packages/trl/trainer/grpo_trainer.py", line 961, in __init__
      su

In [39]:
%%writefile train_grpo.py
import os
import re
import random
import logging
import warnings
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

# Disable progress bars and filter warnings
os.environ["HF_DATASETS_DISABLE_PROGRESS_BAR"] = "1"
warnings.filterwarnings("ignore")

# Configure logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GRPO_Training")

# Seed setup
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# =====================================================================
# 1. Configuration Class
# =====================================================================
@dataclass
class TrainingConfig:
    model_name: str = "Qwen/Qwen3-0.6B"
    output_dir: str = "./grpo_finetuned_model"
    num_train_epochs: int = 5
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 32
    learning_rate: float = 5e-6
    num_generations: int = 12
    temperature: float = 0.8
    max_new_tokens: int = 400
    max_prompt_length: int = 512
    train_split_ratio: float = 0.8
    eval_steps: int = 20
    save_steps: int = 20
    logging_steps: int = 20
    save_total_limit: int = 3
    seed: int = 42
    use_8bit: bool = False


# =====================================================================
# 2. GSM8KRewardSignal Class
# =====================================================================
class GSM8KRewardSignal:
    def __init__(self):
        self.logger = logging.getLogger(self.__class__.__name__)

    def extract_numerical_answer(self, text: str) -> Optional[float]:
        if "####" in text:
            answer = text.split("####")[-1].strip()
            answer = answer.replace(",", "").replace("$", "")
            try:
                return float(answer)
            except Exception:
                pass

        patterns = [
            r"(?:The answer is|answer:|Answer:)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"(?:equals?|=)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"(?:total|sum|result|Total|Final answer)\s*(?:is|:|=)?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"Therefore,?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
            if matches:
                try:
                    clean_match = matches[-1].replace("$", "").replace(",", "")
                    return float(clean_match)
                except ValueError:
                    continue

        numbers = re.findall(r"[+-]?\d+(?:\.\d+)?", text)
        if numbers:
            try:
                return float(numbers[-1])
            except ValueError:
                pass
        return None

    def analyze_response_quality(self, response: str) -> Dict[str, any]:
        calculation_indicators = ["=", "+", "-", "*", "/", "multiply", "divide", "add", "subtract"]
        has_calculation = any(op in response.lower() for op in calculation_indicators)

        sentences = [s.strip() for s in re.split(r"[.!?]+", response) if s.strip()]
        sentence_count = len(sentences)
        has_steps = (sentence_count >= 2) or ("step" in response.lower())

        reasoning_words = [
            "therefore", "because", "since", "so", "thus", "hence",
            "first", "then", "next", "finally", "must", "need"
        ]
        has_reasoning = any(word in response.lower() for word in reasoning_words)
        has_numbers = bool(re.search(r"\d", response))
        response_length = len(response)

        return {
            "has_calculation": has_calculation,
            "has_steps": has_steps,
            "has_reasoning": has_reasoning,
            "has_numbers": has_numbers,
            "response_length": response_length,
            "sentence_count": sentence_count,
        }

    def compute_unparseable_reward(self, response: str, correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
        response_length = quality.get("response_length", 0)
        has_calculation = quality.get("has_calculation", False)
        has_steps = quality.get("has_steps", False)
        has_numbers = quality.get("has_numbers", False)

        if response_length < 20:
            return 0.0

        reward = 0.05
        if has_calculation:
            reward += 0.05
        if has_steps:
            reward += 0.05
        if has_numbers:
            reward += 0.05
        if response_length > 100:
            reward += 0.05
        if response_length > 200:
            reward += 0.05

        return min(0.3, reward)

    def compute_correct_reward(self, response: str, predicted: float, correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
        reward = 1.0
        if quality.get("has_steps", False):
            reward += 0.1
        if quality.get("response_length", 0) > 100:
            reward += 0.1
        if quality.get("has_reasoning", False):
            reward += 0.05
        if quality.get("has_calculation", False):
            reward += 0.05

        return min(reward, 1.3)

    def compute_wrong_reward(self, response: str, predicted: float, correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
        if correct_answer != 0:
            relative_error = abs(predicted - correct_answer) / abs(correct_answer)
        else:
            relative_error = abs(predicted - correct_answer)

        if relative_error < 0.01:
            reward = 0.9
        elif relative_error <= 0.10:
            reward = 0.3
        elif relative_error <= 0.30:
            reward = 0.1
        elif (correct_answer != 0) and (0.1 <= (predicted / correct_answer) <= 10) and (relative_error > 0.5):
            reward = 0.3
        else:
            reward = 0.1

        if quality.get("has_calculation", False) and quality.get("has_steps", False):
            reward += 0.1

        reward = max(0.1, reward)
        return min(0.9, reward)

    def compute_reward(self, response: str, correct_answer: float, question: str = None) -> float:
        predicted = self.extract_numerical_answer(response)
        quality = self.analyze_response_quality(response)

        if predicted is None:
            return self.compute_unparseable_reward(response, correct_answer, quality, question)
        elif abs(predicted - correct_answer) < 0.01:
            return self.compute_correct_reward(response, predicted, correct_answer, quality, question)
        else:
            return self.compute_wrong_reward(response, predicted, correct_answer, quality, question)


# Global instances
reward_model = GSM8KRewardSignal()
prompt2ans = {}
config = TrainingConfig()


# =====================================================================
# 3. Reward Function for GRPO
# =====================================================================
def compute_rewards(prompts: List[str], completions: List[str], **kwargs):
    rewards = []

    seen = set()
    unique_prompts = []
    for p in prompts:
        if p not in seen:
            unique_prompts.append(p)
            seen.add(p)

    for unique_prompt in unique_prompts:
        prompt_indices = [idx for idx, p in enumerate(prompts) if p == unique_prompt]
        group_completions = [completions[idx] for idx in prompt_indices]
        correct_answer, question = prompt2ans.get(unique_prompt, (0.0, "Unknown question"))

        group_rewards = [
            reward_model.compute_reward(completion, correct_answer, question)
            for completion in group_completions
        ]

        # Group normalization (mean-centering)
        if len(group_rewards) > 1:
            mean_reward = sum(group_rewards) / len(group_rewards)
            normalized_rewards = [r - mean_reward for r in group_rewards]
        else:
            normalized_rewards = [0.0]

        for idx, norm_reward in zip(prompt_indices, normalized_rewards):
            if len(rewards) <= idx:
                rewards.extend([0.0] * (idx - len(rewards) + 1))
            rewards[idx] = norm_reward

    return rewards


# =====================================================================
# 4. Dataset Preparation
# =====================================================================
def prepare_dataset(config: TrainingConfig) -> Tuple[Dataset, Dataset]:
    raw_data = load_dataset("openai/gsm8k", "main", split="train")

    split_dataset = raw_data.train_test_split(
        train_size=config.train_split_ratio,
        seed=config.seed,
        shuffle=True
    )
    train_data = split_dataset["train"]
    eval_data = split_dataset["test"]

    def extract_answer(answer_text: str) -> float:
        if "####" in answer_text:
            ans_str = answer_text.split("####")[-1].strip().replace(",", "").replace("$", "")
            try:
                return float(ans_str)
            except ValueError:
                pass
        numbers = re.findall(r"[+-]?\d+(?:\.\d+)?", answer_text.replace(",", ""))
        return float(numbers[-1]) if numbers else 0.0

    def format_example(example):
        question = example["question"].strip()
        answer_text = example["answer"]
        prompt = (
            f"Question: {question}\n\n"
            f"Let's solve this step-by-step and find the numerical answer:\n"
        )
        return {
            "prompt": prompt,
            "question": question,
            "answer": extract_answer(answer_text),
            "answer_text": answer_text,
        }

    train_dataset = train_data.map(format_example)
    eval_dataset = eval_data.map(format_example)
    return train_dataset, eval_dataset


# =====================================================================
# 5. Main Multi-GPU Execution
# =====================================================================
def main():
    global prompt2ans, config

    train_dataset, eval_dataset = prepare_dataset(config)

    # Populate answer lookup dictionary
    for item in train_dataset:
        prompt2ans[item["prompt"]] = (item["answer"], item["question"])

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name,
        trust_remote_code=True,
        enable_thinking=False
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # Model
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        attn_implementation="sdpa",
        dtype=torch.float32,
        trust_remote_code=True
    )

    # GRPO Config
    # generation_batch_size is set to 24 (must be divisible by num_generations=12)
    grpo_config = GRPOConfig(
        output_dir=config.output_dir,
        num_train_epochs=config.num_train_epochs,
        per_device_train_batch_size=config.per_device_train_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        generation_batch_size=24,
        learning_rate=config.learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=0.1,
        max_grad_norm=1.0,
        num_generations=config.num_generations,
        temperature=config.temperature,
        logging_steps=config.logging_steps,
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        save_total_limit=config.save_total_limit,
        seed=config.seed,
        fp16=True,
        bf16=False,
        remove_unused_columns=False,
        push_to_hub=False,
        report_to=[],
    )

    # GRPOTrainer
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=compute_rewards,
        args=grpo_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    print("🚀 Starting GRPO training on Multi-GPU...")
    trainer.train()


if __name__ == "__main__":
    main()

Overwriting train_grpo.py


In [40]:
!accelerate launch --multi_gpu --num_processes=2 train_grpo.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/lib/python3.13/dist-packages/accelerate/utils/launch.py:236: UserWarning: Port `29500` is already in use. Accelerate will attempt to launch in a standalone-like mode by finding an open port automatically for this session. If this current attempt fails, or for more control in future runs, please specify a different port (e.g., `--main_process_port <your_chosen_port>`) or use `--main_process_port 0` for automatic selection in your launch command or Accelerate config file.
  warnings.warn(
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HE

## Okay after various approach in the notebook . we fail !!!
### mainly the error is insufficent memory and that considering things just taking on 0.6B model on 8k dataset with 12 generation and training is not feasible and also this gpu are of old architecture to 
# > IN future if i get chance definately try !! 
> if u have gpu definately try and share notebook 